# 🦾 Capstone C4 · Fine-tune a real Vision-Language-Action model and compare it with yours

**Capstone · stage 4 of 4** &nbsp;|&nbsp; ⏱ ~30 min hands-on + **1–5 h of GPU training** &nbsp;|&nbsp; 🖥️ needs a bigger GPU: Colab **A100 or L4** (Colab Pro) or any cloud GPU with ≥ 24 GB

You built the parts of a robot foundation model yourself: a flow-matching action expert (C1), pretrained vision features (C2) and a world model (C3). Now you'll use the **real industry stack**:

| Component | SmolVLA (this notebook) | Physical Intelligence π0 | NVIDIA GR00T N1.x |
|---|---|---|---|
| Vision + language backbone | SmolVLM2 (SigLIP + SmolLM2), **frozen** | PaliGemma 3B | Eagle VLM |
| Action generator | ~100 M-param transformer **action expert**, **flow matching** | 300 M action expert, flow matching | diffusion transformer, flow matching |
| Output | chunk of 50 actions | chunk of 50 | chunk of 16 |
| Library | Hugging Face **LeRobot** | openpi (also in LeRobot) | Isaac-GR00T (also in LeRobot) |

**Your research question:** *on a narrow task like PushT, does a 450 M-parameter pretrained VLA beat your 6 M-parameter specialist, and is it worth the GPU hours?* There is no "right" answer. A careful, honest comparison is the portfolio piece.

> ✅ **Verified for this course (September 2026):** LeRobot **0.6.1** + `lerobot/smolvla_base`: the training and evaluation commands below ran end-to-end on the PushT dataset and simulator (a 2-step smoke test). **Not verified:** how good the model gets after a full run. That's your experiment.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import os, sys, json, subprocess, importlib.util, time
from pathlib import Path
import numpy as np
FAST_DEV_RUN = os.environ.get("CAPSTONE_FAST") == "1"      # course authors' quick self-test switch

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if isinstance(obj, dict):
        return any(_has_blank(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return any(_has_blank(v) for v in obj)
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
CHALLENGES["rename"] = dict(title="Connect the dataset camera to the model",
    reference={"observation.image": "observation.images.camera1"},
    test=lambda m: (isinstance(m, dict) and m.get("observation.image") == "observation.images.camera1",
                    "The dataset key <code>observation.image</code> should map to the model's first camera slot."),
    hint="The pretrained SmolVLA expects cameras named <code>observation.images.camera1</code>, <code>…camera2</code>, <code>…camera3</code>. PushT has one camera called <code>observation.image</code>.",
    why="Mismatched observation names are the most common reason a fine-tune silently ignores its camera. LeRobot's <code>--rename_map</code> fixes it without touching the dataset.")

CHALLENGES["epochs"] = dict(title="How many times will the model see the data?",
    reference=lambda steps, batch_size, num_frames: steps * batch_size / num_frames,
    cases=[(20000, 64, 25650), (3000, 32, 25650)],
    hint="Each step shows <code>batch_size</code> frames. Total frames shown ÷ dataset size = epochs.",
    why="Budgeting in epochs lets you compare runs with different batch sizes and spot under- or over-training before you pay for GPU hours.")

CHALLENGES["action_steps"] = dict(title="How many actions to execute per plan?",
    reference=8,
    hint="PushT runs at 10 steps per second. Executing 0.8 seconds of each plan before re-planning means 10 × 0.8 actions.",
    why="SmolVLA predicts 50 actions (5 s at 10 Hz) by default, which is far too long to commit to for a contact-rich task. Matching C1's 8-step receding horizon also makes the comparison fair.")

QUIZZES["frozen"] = dict(q="During this fine-tune, which part of SmolVLA's ~450 M parameters actually learns?",
    options=["All of it", "Only the ~100 M-parameter action expert (plus small projections); the vision-language model stays frozen", "Only the vision encoder"],
    answer=1, explain="SmolVLA's defaults are <code>freeze_vision_encoder=True</code> and <code>train_expert_only=True</code>. The pretrained VLM keeps its general knowledge while the action expert adapts, exactly like your C2 setup with frozen DINOv2.")
QUIZZES["resolution"] = dict(predict=True, q="LeRobot's PushT simulator renders observations at 384×384 by default, but the training images are 96×96. What happens if you evaluate without changing it?",
    options=["Nothing: higher resolution is always better", "A train/test mismatch: the model sees sharper, different-looking images than in training, which can quietly lower its score", "The evaluation crashes"],
    answer=1, explain="Evaluating on inputs rendered differently from training is <b>distribution shift</b>. We set <code>--env.observation_height=96 --env.observation_width=96</code> so evaluation matches the data. Always compare the images your model trains on with the ones it's tested on.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
## 0 · Check your GPU and pick a budget 💰
Official Hugging Face guidance: **20,000 steps at batch size 64 take about 5 hours on an A100.** Use the table to plan.

| Budget | Steps × batch | Epochs of PushT | A100 time (scaled from HF's figure) |
|---|---|---|---|
| 🟢 Smoke test | 20 × 8 | 0.006 | a few minutes, only checks that it runs |
| 🟡 Budget run | 3,000 × 32 | ~3.7 | ~25–40 min |
| 🔵 Full run | 20,000 × 64 | ~50 | ~5 h |

Times other than HF's 5 h figure are proportional estimates. The training log prints the real speed, so re-plan after the first few minutes.

In [ ]:
import shutil
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else ""
print(gpu or "No NVIDIA GPU visible. Fine for reading along, not for training (Runtime → Change runtime type).")

### 🧩 Challenge 1 · How many times will the model see the data?

In [ ]:
def epochs(steps, batch_size, num_frames):
    return ___      # 🧩 frames shown ÷ dataset size

epochs = check("epochs", epochs)
BUDGET = "smoke" if FAST_DEV_RUN else "budget"          # ← choose "smoke", "budget" or "full"
STEPS, BATCH = {"smoke": (20, 8), "budget": (3000, 32), "full": (20000, 64)}[BUDGET]
if FAST_DEV_RUN: STEPS, BATCH = 2, 2
print(f"{BUDGET}: {STEPS} steps × batch {BATCH} = {epochs(STEPS, BATCH, 25650):.2f} epochs of PushT")

<details><summary>🤔 <b>Need a hint?</b></summary>

Multiply steps by batch size, divide by the number of frames.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return steps * batch_size / num_frames      # 🧩 frames shown ÷ dataset size</pre>

</details>

---
## 1 · Install LeRobot (pinned) 📦
This installs the exact version tested for the course. It may replace Colab's PyTorch with a compatible one (LeRobot 0.6.1 needs `torch < 2.12`). If you see *"restart required"*, use **Runtime → Restart session**, then continue from the next cell.

In [ ]:
if importlib.util.find_spec("lerobot") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lerobot[smolvla,pusht]==0.6.1"])
import lerobot, torch
print("lerobot", lerobot.__version__, "· torch", torch.__version__, "· CUDA", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

---
## 2 · Look at the data the way LeRobot sees it 🔎
Same 206 demonstrations as C1–C3, published in LeRobot format (`lerobot/pusht`), with videos, a 2-D state (the pusher position) and a **language instruction**.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset("lerobot/pusht", video_backend="pyav" if FAST_DEV_RUN else None)
print(ds)
sample = ds[100]
for k, v in sample.items():
    print(f"  {k:>20}: {tuple(v.shape) if hasattr(v, 'shape') else v}")
import matplotlib.pyplot as plt
plt.figure(figsize=(2.5, 2.5)); plt.imshow(sample["observation.image"].permute(1, 2, 0)); plt.axis("off")
plt.title("what the VLA sees"); plt.show()
print("instruction:", sample["task"])

The pretrained model was trained with **three** camera slots called `observation.images.camera1/2/3`. PushT has **one** camera called `observation.image`.

### 🧩 Challenge 2 · Connect the dataset camera to the model

In [ ]:
RENAME_MAP = {"observation.image": ___}     # 🧩 dataset key → model key
RENAME_MAP = check("rename", RENAME_MAP)

<details><summary>🤔 <b>Need a hint?</b></summary>

The model's first camera slot.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>RENAME_MAP = {"observation.image": "observation.images.camera1"}     # 🧩 dataset key → model key</pre>

</details>

---
## 3 · What will actually train? 🧠

In [ ]:
quiz("frozen")

### 🧩 Challenge 3 · How many actions to execute per plan?

In [ ]:
N_ACTION_STEPS = ___            # 🧩 10 Hz × 0.8 s
N_ACTION_STEPS = check("action_steps", N_ACTION_STEPS)

<details><summary>🤔 <b>Need a hint?</b></summary>

Steps per second × seconds.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>N_ACTION_STEPS = 8            # 🧩 10 Hz × 0.8 s</pre>

</details>

---
## 4 · Fine-tune 🏋️
We build the command from your choices so you can read every flag:

| Flag | Why |
|---|---|
| `--policy.path=lerobot/smolvla_base` | start from the pretrained VLA |
| `--dataset.repo_id=lerobot/pusht` | our demonstrations |
| `--rename_map=…` | connect the camera (challenge 2) |
| `--policy.n_action_steps=8` | receding horizon (challenge 3); training still predicts the full 50-action chunk |
| `--policy.push_to_hub=false` | keep checkpoints local until you decide to share |
| `--save_freq` | Colab can disconnect, so save often |

💾 **Protect long runs:** set `OUTPUT_DIR` to a folder in Google Drive (`/content/drive/MyDrive/…` after mounting it from the Files panel). To resume after a disconnect, re-run this cell with `RESUME = True`.

In [ ]:
OUTPUT_DIR = Path("capstone_outputs/c4_smolvla")
RESUME = False
cmd = ["lerobot-train",
       "--policy.path=lerobot/smolvla_base",
       "--dataset.repo_id=lerobot/pusht",
       f"--rename_map={json.dumps(RENAME_MAP)}",
       f"--policy.n_action_steps={N_ACTION_STEPS}",
       f"--policy.device={DEVICE}",
       "--policy.push_to_hub=false",
       f"--batch_size={BATCH}", f"--steps={STEPS}",
       f"--save_freq={max(1, min(1000, STEPS))}", f"--log_freq={max(1, min(100, STEPS))}",
       f"--output_dir={OUTPUT_DIR}", "--job_name=c4_smolvla_pusht"]
if FAST_DEV_RUN:
    cmd += ["--num_workers=0", "--dataset.video_backend=pyav"]
if RESUME:
    cmd = ["lerobot-train", f"--config_path={OUTPUT_DIR}/checkpoints/last/pretrained_model/train_config.json", "--resume=true"]
if OUTPUT_DIR.exists() and not RESUME:
    print(f"{OUTPUT_DIR} already exists. Delete it, pick a new OUTPUT_DIR, or set RESUME = True.");
else:
    print(" \\\n  ".join(cmd)); t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if any(key in line for key in ("step:", "INFO", "Error", "error")) and "torchcodec" not in line:
            print(line.rstrip()[-220:])
    proc.wait(); print(f"exit code {proc.returncode} · {(time.time() - t0) / 60:.1f} min")
    TRAIN_MINUTES = (time.time() - t0) / 60

---
## 5 · Evaluate on the same 50 scenes as C1–C3 🎯
`lerobot-eval --seed=100000` with 50 episodes uses seeds 100000–100049, **the same start states as your own policies**. LeRobot's `avg_max_reward` is the same **score** you computed in C1 (best coverage ÷ 0.95, capped at 1).

In [ ]:
quiz("resolution")

In [ ]:
CKPT = OUTPUT_DIR / "checkpoints" / "last" / "pretrained_model"
EVAL_DIR = Path("capstone_outputs/c4_eval")
N_EPISODES, EVAL_BATCH = (1, 1) if FAST_DEV_RUN else (50, 10)
eval_cmd = ["lerobot-eval", f"--policy.path={CKPT}", "--env.type=pusht",
            "--env.observation_height=96", "--env.observation_width=96",       # match the training images
            f"--rename_map={json.dumps(RENAME_MAP)}", f"--policy.device={DEVICE}",
            f"--eval.n_episodes={N_EPISODES}", f"--eval.batch_size={EVAL_BATCH}", "--seed=100000",
            f"--output_dir={EVAL_DIR}"]
t0 = time.time()
out = subprocess.run(eval_cmd, capture_output=True, text=True)
print(f"exit code {out.returncode} · {(time.time() - t0) / 60:.1f} min")
if out.returncode:
    print(out.stdout[-3000:], out.stderr[-3000:])
info = json.loads((EVAL_DIR / "eval_info.json").read_text())
per_episode = np.array(info["per_task"][0]["metrics"]["max_rewards"], float)
r = np.random.default_rng(0); boots = per_episode[r.integers(0, len(per_episode), (2000, len(per_episode)))].mean(1)
smolvla = dict(label="SmolVLA fine-tuned", score=float(per_episode.mean()), ci95=[float(np.quantile(boots, .025)), float(np.quantile(boots, .975))],
               success_rate=info["overall"]["pc_success"] / 100, episodes=len(per_episode), steps=STEPS, batch=BATCH)
print(json.dumps(smolvla, indent=1))
(Path("capstone_outputs") / "c4_results.json").write_text(json.dumps(smolvla, indent=1))

---
## 6 · The comparison that goes in your report 📊
Upload `c1_results.json`, `c2_results.json` and `c3_results.json` from earlier stages into `capstone_outputs/` (Colab Files panel) if they aren't already there.

In [ ]:
rows = []
for name, key in [("c1_results.json", "flow"), ("c1_results.json", "regression"), ("c2_results.json", "policy"), ("c2_results.json", "modular"), ("c3_results.json", "wm_pick")]:
    p = Path("capstone_outputs") / name
    if p.exists():
        d = json.loads(p.read_text())
        if key in d: rows.append(d[key])
rows.append(smolvla)
labels = [r["label"] for r in rows]; means = [r["score"] for r in rows]
err = [[m - r["ci95"][0] for m, r in zip(means, rows)], [r["ci95"][1] - m for m, r in zip(means, rows)]]
plt.figure(figsize=(7, 0.6 + 0.5 * len(rows)))
plt.barh(labels, means, xerr=err, color="#3b8ea5", capsize=4)
c1_file = Path("capstone_outputs") / "c1_results.json"
human = json.loads(c1_file.read_text())["human_reference"] if c1_file.exists() else None
if human: plt.axvline(human, ls="--", c="k"); plt.text(human, -0.6, " human demos", fontsize=8)
plt.xlim(0, 1); plt.xlabel("PushT score on the same 50 scenes (95% CI)"); plt.title("specialist vs foundation model"); plt.show()

### How to write this up honestly 📝
* **Report compute:** GPU type, minutes of training and evaluation, parameter counts (6.4 M vs 450 M total / 100 M trained).
* **Name the confounds:** SmolVLA saw 2-D pusher position plus images; C1 saw the full state (block pose). C2's two vision pipelines are the fairest comparisons, and C2-modular also used pose labels during training.
* **Overlapping intervals ⇒ no winner.** Say so (lab 03).
* **Interpretation, not hype:** a pretrained VLA's advantages (language, many tasks, transfer) don't show on a single 2-D task. What experiment *would* show them? (Hint: LIBERO, below.)

### 🚀 Where to go next
* **Real language conditioning:** LeRobot's LIBERO suites (`--env.type=libero`) have ~10 tasks with different instructions per suite. Fine-tune SmolVLA on a LIBERO dataset and test unseen instruction phrasings. *(Not run for this course; check the LeRobot LIBERO docs for current dataset names.)*
* **Bigger VLAs:** the same `lerobot-train` interface supports `--policy.type` values `pi05` (Physical Intelligence), `groot` (NVIDIA) and `xvla`, with larger GPU needs.
* **Cheaper fine-tuning:** try `--policy.use_peft=true` (LoRA adapters) and compare score per GPU-hour.
* **Share it:** `huggingface-cli upload <your-username>/smolvla_pusht capstone_outputs/c4_smolvla/checkpoints/last/pretrained_model`. Store your token in Colab's 🔑 Secrets panel, never in a cell.

In [ ]:
progress_report()